# Point Cloud Benchmarking — Quick Demo

This notebook walks through the core functionality of the framework:
1. Generate synthetic point clouds
2. Run all 5 metrics
3. Visualise error distributions
4. Full pipeline evaluation

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from src.processing.io_utils import generate_synthetic_sphere, generate_synthetic_cube
from src.metrics.chamfer import chamfer_distance
from src.metrics.hausdorff import hausdorff_distance
from src.metrics.fscore import f_score
from src.metrics.normal_consistency import normal_consistency
from src.metrics.emd import earth_movers_distance
from src.indexing.kdtree import KDTreeIndex

print('All imports OK')

## 1. Generate Synthetic Data

In [ ]:
N = 5000
gt = generate_synthetic_sphere(N, radius=1.0, seed=42)
pred_noisy = generate_synthetic_sphere(N, radius=1.0, noise_std=0.02, seed=43)
pred_offset = gt + np.array([0.05, 0.0, 0.0])

print(f'Ground truth:  {gt.shape}')
print(f'Noisy pred:    {pred_noisy.shape}')
print(f'Offset pred:   {pred_offset.shape}')

## 2. Chamfer Distance

In [ ]:
cd_noisy = chamfer_distance(pred_noisy, gt)
cd_offset = chamfer_distance(pred_offset, gt)

print('--- Noisy ---')
print(f"  CD mean:    {cd_noisy['chamfer_mean']:.6f}")
print(f"  CD max:     {cd_noisy['chamfer_max']:.6f}")
print()
print('--- Offset ---')
print(f"  CD mean:    {cd_offset['chamfer_mean']:.6f}")
print(f"  CD max:     {cd_offset['chamfer_max']:.6f}")

## 3. Hausdorff Distance

In [ ]:
hd_noisy = hausdorff_distance(pred_noisy, gt)
hd_offset = hausdorff_distance(pred_offset, gt)

print(f"Noisy  HD (symmetric): {hd_noisy['hausdorff']:.6f}")
print(f"Offset HD (symmetric): {hd_offset['hausdorff']:.6f}")

## 4. F-Score at Multiple Thresholds

In [ ]:
thresholds = [0.005, 0.01, 0.02, 0.05, 0.1]
fs = f_score(pred_noisy, gt, thresholds=thresholds)

print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print('-' * 45)
for entry in fs['results']:
    print(f"{entry['threshold']:>10.3f} {entry['precision']:>10.4f} {entry['recall']:>10.4f} {entry['f1']:>10.4f}")

## 5. Normal Consistency

In [ ]:
# For a sphere, normals = normalised point positions
gt_normals = gt / np.linalg.norm(gt, axis=1, keepdims=True)
pred_normals = pred_noisy / np.linalg.norm(pred_noisy, axis=1, keepdims=True)

nc = normal_consistency(pred_noisy, gt, pred_normals, gt_normals)
print(f"Normal Consistency (mean): {nc['normal_consistency_mean']:.4f}")
print(f"Normal Consistency (std):  {nc['normal_consistency_std']:.4f}")

## 6. Earth Mover's Distance

In [ ]:
emd = earth_movers_distance(pred_noisy, gt, max_points=1024)
print(f"EMD (mean cost): {emd['emd']:.6f}")
print(f"EMD (sqrt):      {emd['emd_sqrt']:.6f}")
print(f"Points used:     {emd['n_points_used']}")

## 7. Error Distribution Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Chamfer per-point
axes[0].hist(cd_noisy['pred_to_gt'], bins=50, alpha=0.7, color='steelblue')
axes[0].set_title('Chamfer: Pred→GT Distances')
axes[0].set_xlabel('Distance')
axes[0].set_ylabel('Count')

# Normal consistency per-point
axes[1].hist(nc['per_point_consistency'], bins=50, alpha=0.7, color='seagreen')
axes[1].set_title('Normal Consistency (per point)')
axes[1].set_xlabel('|cos θ|')
axes[1].set_ylabel('Count')

# F-score vs threshold
taus = [r['threshold'] for r in fs['results']]
f1s = [r['f1'] for r in fs['results']]
axes[2].plot(taus, f1s, 'o-', color='coral', linewidth=2)
axes[2].set_title('F1 vs Threshold')
axes[2].set_xlabel('Threshold')
axes[2].set_ylabel('F1 Score')
axes[2].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 8. Full Pipeline Evaluation

In [ ]:
from src.evaluation.evaluator import PointCloudEvaluator
from src.evaluation.report import generate_report

config = {
    'preprocessing': {'enabled': False},
    'metrics': {
        'chamfer': {'enabled': True, 'squared': False},
        'hausdorff': {'enabled': True, 'percentile': 100},
        'fscore': {'enabled': True, 'thresholds': [0.01, 0.05]},
        'normal_consistency': {'enabled': False},
        'emd': {'enabled': True, 'max_points': 1024},
    }
}

evaluator = PointCloudEvaluator(config)
results = evaluator.evaluate(pred_noisy, gt, preprocess=False)

report = generate_report(results)
print(report)

---
**Next steps**: Try loading your own `.ply`/`.npy` files using `load_point_cloud()`, or run the CLI:
```bash
python evaluate.py evaluate --pred data/predicted/sphere.npy --gt data/ground_truth/sphere.npy
```